# Notebook 04 — Secondary Structure and Ramachandran Plots

**Module:** 11 — Structural Biology  
**Tier:** 3 — Survey  
**Notebook:** 04 of 08  
**Time estimate:** 75 minutes

> The Ramachandran plot is the most famous plot in structural biology.
> It shows, in two dimensions, which backbone conformations are physically allowed.
> Building it from scratch — from atomic coordinates to dihedral angles — is the
> computational core of this module.

---
## Step 1 — Motivation

Every crystallography paper reports Ramachandran statistics as a quality indicator.
The DSSP algorithm assigns secondary structure from backbone geometry. Both rest on
computing dihedral angles from atomic coordinates — a fundamental 3D geometry operation
that appears in structure validation, structure prediction evaluation (pLDDT), and
molecular dynamics analysis.

---
## Step 2 — Intuition

**Ramachandran plot:**  
Plot every residue as a point at (φ, ψ). Sterically allowed regions are densely
populated; forbidden regions (where atoms clash) are empty.

Three main allowed regions:
- **α-helical region:** φ ≈ −60°, ψ ≈ −40° (top-left quadrant)
- **β-sheet region:** φ ≈ −120°, ψ ≈ +130° (bottom-left quadrant)
- **Left-handed helix:** φ ≈ +60°, ψ ≈ +40° (small, only Gly)

**DSSP algorithm:**  
Assigns secondary structure based on backbone H-bond patterns:
- α-helix: H-bond between C=O(i) and N-H(i+4)
- β-sheet: H-bonds between strands
- Uses the Kabsch-Sander energy criterion: 
  $E_{\text{H-bond}} < -0.5$ kcal/mol to define a bond

---
## Step 3 — Biological Background

**Why are most (φ, ψ) combinations forbidden?**  
Steric clash between the carbonyl oxygen (C=O) and the side-chain atoms of adjacent
residues. Glycine (no side chain) has the largest allowed region — it can access
left-handed helix conformations impossible for other amino acids.

**Proline is special:** The cyclic pyrrolidine ring constrains φ to approximately
−60° and prevents the backbone N from donating an H-bond. This is why proline
is a helix-breaker and a common turn initiator.

**Secondary structure composition of proteins:**  
Globular proteins average ~30% α-helix, ~25% β-sheet, ~45% loops/turns,
but this varies enormously by fold class (all-α, all-β, α/β, α+β).

---
## Step 4 — Mathematical Explanation

**Dihedral angle computation via cross products:**

Given atoms $P_1, P_2, P_3, P_4$, let:
$$\mathbf{b}_1 = P_2 - P_1, \quad \mathbf{b}_2 = P_3 - P_2, \quad \mathbf{b}_3 = P_4 - P_3$$
$$\mathbf{n}_1 = \mathbf{b}_1 \times \mathbf{b}_2, \quad \mathbf{n}_2 = \mathbf{b}_2 \times \mathbf{b}_3$$
$$\mathbf{m}_1 = \mathbf{n}_1 \times \hat{\mathbf{b}}_2$$
$$\theta = \text{atan2}(\mathbf{m}_1 \cdot \mathbf{n}_2,\; \mathbf{n}_1 \cdot \mathbf{n}_2)$$

Result is in radians; convert to degrees for Ramachandran plots.

**φ for residue $i$:** dihedral C′(i-1) – N(i) – Cα(i) – C′(i)  
**ψ for residue $i$:** dihedral N(i) – Cα(i) – C′(i) – N(i+1)

**DSSP hydrogen bond energy:**
$$E = 0.084 \left(\frac{1}{r_{ON}} + \frac{1}{r_{CH}} - \frac{1}{r_{OH}} - \frac{1}{r_{CN}}\right) \times 332 \text{ kcal/mol}$$

where distances are in Ångströms. $E < -0.5$ kcal/mol = H-bond.

In [ ]:
# Step 6 — Python: Dihedral angle computation and Ramachandran plot
import numpy as np
import matplotlib.pyplot as plt

def dihedral_angle(p1, p2, p3, p4):
    """Compute dihedral angle (degrees) for four 3D points p1–p4."""
    b1 = np.array(p2) - np.array(p1)
    b2 = np.array(p3) - np.array(p2)
    b3 = np.array(p4) - np.array(p3)
    # Normal vectors to the two planes
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    # Handle degenerate cases
    if np.linalg.norm(n1) < 1e-9 or np.linalg.norm(n2) < 1e-9:
        return np.nan
    n1 /= np.linalg.norm(n1)
    n2 /= np.linalg.norm(n2)
    b2_hat = b2 / np.linalg.norm(b2)
    m1 = np.cross(n1, b2_hat)
    x = np.dot(n1, n2)
    y = np.dot(m1, n2)
    return np.degrees(np.arctan2(y, x))

# ---- Simulate backbone atom coordinates for a mixed structure ----
# We'll generate ideal helix (φ≈-60, ψ≈-40) and sheet (φ≈-120, ψ≈130) segments
# plus a random coil region
def backbone_from_dihedrals(phi_list, psi_list, bond_len_ca_c=1.52,
                              bond_len_c_n=1.33, bond_len_n_ca=1.46,
                              angle_ca_c_n=np.radians(116.0),
                              angle_c_n_ca=np.radians(122.0),
                              angle_n_ca_c=np.radians(111.0)):
    """Build backbone atom positions from φ/ψ angles (simplified nerf algorithm)."""
    # Start with first three atoms at known positions
    N  = np.array([0.0, 0.0, 0.0])
    CA = np.array([bond_len_n_ca, 0.0, 0.0])
    C  = np.array([
        bond_len_n_ca + bond_len_ca_c * np.cos(np.pi - angle_n_ca_c),
        bond_len_ca_c * np.sin(np.pi - angle_n_ca_c), 0.0
    ])
    coords = {'N': [N], 'CA': [CA], 'C': [C]}

    for phi, psi in zip(phi_list, psi_list):
        phi_rad = np.radians(phi)
        psi_rad = np.radians(psi)
        # Place next N using psi (rotation around CA-C bond)
        prev_c   = coords['C'][-1]
        prev_ca  = coords['CA'][-1]
        prev_n   = coords['N'][-1]
        # Use a simple rotation approach (not full NERF for brevity)
        # For simulation purposes, generate coordinates with approximate geometry
        z_shift = bond_len_c_n * np.cos(np.pi - angle_ca_c_n) * 0.5
        y_shift = bond_len_c_n * np.sin(np.pi - angle_ca_c_n) * 0.5
        new_n  = prev_c + np.array([np.cos(psi_rad), np.sin(psi_rad), z_shift]) * bond_len_c_n
        new_ca = new_n + np.array([np.cos(phi_rad + np.pi), np.sin(phi_rad + np.pi), 0.0]) * bond_len_n_ca
        new_c  = new_ca + np.array([np.cos(phi_rad), np.sin(phi_rad), z_shift * 0.3]) * bond_len_ca_c
        coords['N'].append(new_n)
        coords['CA'].append(new_ca)
        coords['C'].append(new_c)
    return coords

rng = np.random.default_rng(42)

# Simulate structures with known phi/psi distributions
n_helix  = 60
n_sheet  = 40
n_loop   = 50

# Ideal alpha helix: phi ≈ -60 ± 10, psi ≈ -40 ± 10
phi_h = rng.normal(-60, 8, n_helix)
psi_h = rng.normal(-40, 8, n_helix)

# Ideal beta sheet: phi ≈ -120 ± 15, psi ≈ 130 ± 15
phi_b = rng.normal(-120, 12, n_sheet)
psi_b = rng.normal(130, 12, n_sheet)

# Random coil: broadly distributed in allowed regions
# Mix of left-handed (-120 to -30) and extended (+50 to +180)
phi_l = np.concatenate([
    rng.uniform(-150, -30, n_loop // 2),
    rng.normal(60, 15, n_loop // 2)  # left-handed helix (mostly Gly)
])
psi_l = np.concatenate([
    rng.uniform(-60, 60, n_loop // 2),
    rng.normal(40, 15, n_loop // 2)
])

all_phi = np.concatenate([phi_h, phi_b, phi_l])
all_psi = np.concatenate([psi_h, psi_b, psi_l])
ss_type = (['H'] * n_helix + ['E'] * n_sheet + ['L'] * n_loop)

print(f'Total residues: {len(all_phi)}')
print(f'  Helix: {n_helix}, Sheet: {n_sheet}, Loop: {n_loop}')
print(f'Mean phi (helix):  {phi_h.mean():.1f}° ± {phi_h.std():.1f}°')
print(f'Mean psi (helix):  {psi_h.mean():.1f}° ± {psi_h.std():.1f}°')
print(f'Mean phi (sheet):  {phi_b.mean():.1f}° ± {phi_b.std():.1f}°')
print(f'Mean psi (sheet):  {psi_b.mean():.1f}° ± {psi_b.std():.1f}°')

In [ ]:
# Step 7 — Visualization: Ramachandran plot
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

ss_colors = {'H': 'steelblue', 'E': 'tomato', 'L': 'grey'}

# Panel A: Full Ramachandran plot colored by secondary structure
ax = axes[0]
for ss, col in ss_colors.items():
    mask = [s == ss for s in ss_type]
    ax.scatter(all_phi[mask], all_psi[mask], c=col, s=15, alpha=0.6, label=ss)
# Add reference regions
from matplotlib.patches import Ellipse
helix_region  = Ellipse((-60, -40),  width=80, height=80, color='steelblue', alpha=0.1)
sheet_region  = Ellipse((-120, 130), width=80, height=80, color='tomato',    alpha=0.1)
ax.add_patch(helix_region)
ax.add_patch(sheet_region)
ax.axhline(0, color='black', lw=0.5, alpha=0.3)
ax.axvline(0, color='black', lw=0.5, alpha=0.3)
ax.set_xlabel('φ (degrees)')
ax.set_ylabel('ψ (degrees)')
ax.set_title('A. Ramachandran plot\n(colored by secondary structure)')
ax.legend(title='SS type', fontsize=8)
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_aspect('equal')
ax.text(-60, -40, 'α-helix', ha='center', fontsize=9, color='steelblue', fontweight='bold')
ax.text(-120, 130, 'β-sheet', ha='center', fontsize=9, color='tomato', fontweight='bold')

# Panel B: 2D histogram (density Ramachandran plot)
ax = axes[1]
ax.hist2d(all_phi, all_psi, bins=40, cmap='Blues')
ax.axhline(0, color='white', lw=0.5, alpha=0.5)
ax.axvline(0, color='white', lw=0.5, alpha=0.5)
ax.set_xlabel('φ (degrees)')
ax.set_ylabel('ψ (degrees)')
ax.set_title('B. Ramachandran density map\n(bright = many residues)')
ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_aspect('equal')

plt.suptitle('Module 11 NB04: Secondary Structure and Ramachandran Plots', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ramachandran.png', dpi=150, bbox_inches='tight')
plt.show()

# Secondary structure composition
from collections import Counter
comp = Counter(ss_type)
total = len(ss_type)
print('\nSecondary structure composition:')
for ss, count in sorted(comp.items()):
    print(f'  {ss}: {count:3d} ({count/total*100:.0f}%)')

---
## Step 8 — Exercises

1. Implement `dihedral_angle(p1, p2, p3, p4)` from scratch. Verify that for an
   ideal helix with φ = −60°, ψ = −40°, your function returns those values.
2. Modify the simulation so 10 residues have φ = +60°, ψ = +40° (left-handed helix).
   Where do they appear on the Ramachandran plot?
3. What is the DSSP algorithm and what criterion does it use to define a hydrogen bond?
   (Survey question — look up the Kabsch-Sander paper.)
4. A residue in a high-resolution X-ray structure is found in the disallowed region
   of the Ramachandran plot. List two possible explanations.

---
## Step 10 — Quiz

1. What does each point on a Ramachandran plot represent?
2. Why does glycine have a larger allowed region than other amino acids?
3. What are the typical (φ, ψ) values for α-helices and β-sheets?
4. What does it mean for a residue to be in the "generously allowed" region?
5. Why can proline not be in the middle of an α-helix?

---
## Step 12 — Reflection

> *[In one paragraph: explain how Ramachandran statistics are used to validate
> protein structures, and why a high fraction of residues in allowed regions
> is a necessary but not sufficient condition for a correct structure.]*

---
*Next: `05_contact_maps_and_distance_geometry.ipynb`*